# Real MLP training
Research only, not clinical diagnosis. Pima cohort is not representative of the general population. Raw dataset is not synthetic; never use it as demo patient data. See docs/data_card.md. Fixed seed=42; train/validation/test=70/15/15. Test never used for fitting or model selection.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
assert (ROOT/'pyproject.toml').exists()
sys.path.insert(0,str(ROOT))
import matplotlib
matplotlib.use('Agg')
from backend.app.ml.pipeline import *
import matplotlib.pyplot as plt


In [2]:
def train():
    from torch.utils.data import TensorDataset, DataLoader
    import matplotlib.pyplot as plt
    seed_all(); parts=arrays()
    x,y=parts['train']; vx,vy=map(torch.from_numpy,parts['validation'])
    loader=DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(y)),batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(SEED))
    model=MLP(); optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
    loss_fn=nn.BCEWithLogitsLoss(); best=float('inf'); stale=0; history=[]
    for epoch in range(1,201):
        model.train(); total=0.
        for bx,by in loader:
            optimizer.zero_grad(); loss=loss_fn(model(bx),by); loss.backward(); optimizer.step(); total+=loss.item()*len(by)
        model.eval()
        with torch.no_grad(): val=float(loss_fn(model(vx),vy))
        history.append({'epoch':epoch,'train_loss':total/len(y),'validation_loss':val})
        if val < best-1e-5:
            best=val; stale=0
            torch.save({'state_dict':model.state_dict(),'model_version':VERSION,'epoch':epoch,'input_features':FEATURES},ART/'checkpoint.pt')
        else: stale+=1
        if stale>=20: break
    config={'seed':SEED,'max_epochs':200,'early_stopping_patience':20,'min_delta':1e-5,'optimizer':'Adam','learning_rate':0.001,'batch_size':32,'loss':'BCEWithLogitsLoss','threshold':0.5,'architecture':[8,32,16,1],'dropout':0.15,'selection':'minimum validation loss','device':'cpu'}
    save_json(ART/'training_config.json',config)
    pd.DataFrame(history).to_csv(RES/'learning_history.csv',index=False)
    pd.DataFrame(history).set_index('epoch').plot(); plt.ylabel('BCE loss'); plt.tight_layout(); plt.savefig(FIG/'learning_curves.png'); plt.show(); plt.close()
    environment={'python':platform.python_version(),**{p:importlib.metadata.version(p) for p in ['torch','scikit-learn','pandas','numpy','joblib','matplotlib','nbconvert']}}
    save_json(RES/'environment.json',environment)
    names=['checkpoint.pt','preprocessing.joblib','preprocessing_config.json','training_config.json']
    save_json(ART/'manifest.json',{'model_version':VERSION,'dataset_sha256':sha(ROOT/'data/raw/pima.csv'),'sha256':{p:sha(ART/p) for p in names},'environment':environment,'use':'research only; not clinical diagnosis'})
    print('Epochs executed:',len(history),'best validation loss:',best)
    return pd.DataFrame(history)


In [3]:
history=train()
display(history.tail())

Epochs executed: 44 best validation loss: 0.5023338794708252


D:\TEMP\ipykernel_19596\1443002556.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pd.DataFrame(history).set_index('epoch').plot(); plt.ylabel('BCE loss'); plt.tight_layout(); plt.savefig(FIG/'learning_curves.png'); plt.show(); plt.close()


,epoch,train_loss,validation_loss
39,40,0.409517,0.517443
40,41,0.412769,0.516888
41,42,0.416007,0.517431
42,43,0.423930,0.520485
43,44,0.408441,0.521356


Limitations: small cohort, female Pima ancestry age >=21; historical diagnostic labels, missingness and potential selection bias. Single split has high uncertainty. Brier/reliability curve assess uncalibrated scores; no clinical threshold or external validation. Baselines use identical train-only preprocessing and held-out test; DL need not outperform RF. False positives/negatives are exported for error review without copying patient rows.